In [15]:
import pandas as pd 
inventory = pd.read_parquet(
    r"C:\Users\FEYZA KARA\Downloads\surplus-solutions-showcase-main\surplus-solutions-showcase-main\market_data_output_15d\inventory_snapshot.parquet"
)

sales = pd.read_parquet(
    r"C:\Users\FEYZA KARA\Downloads\surplus-solutions-showcase-main\surplus-solutions-showcase-main\market_data_output_15d\sales_3y.parquet"
)
inventory.head() 

,snapshot_ts,store_id,sku,product_name,on_hand_qty,expiry_date,lot_id,purchased_date,unit_cost_at_lot
0,2026-02-19 01:28:57,X001,SKU00002,TOPITOP MEYVELİ KAVANOZ 120 Lİ,24,2026-03-13,F4096A57C7ED,2025-08-05,3.17
1,2026-02-19 01:28:57,X001,SKU00003,MKA 175G SOFT CHOC 16CA,48,2026-05-01,A0449612B5BF,2025-07-05,4.95
2,2026-02-19 01:28:57,X001,SKU00004,Jelibon Sour Patch Kids 80 gr,144,2026-05-25,0A8610087ACD,2025-09-25,2.74
3,2026-02-19 01:28:57,X001,SKU00005,Tofita Vişne 47gr,48,2026-07-05,404BA1ECEB1A,2025-09-28,2.38
4,2026-02-19 01:28:57,X001,SKU00005,Tofita Vişne 47gr,36,2026-05-19,70A45BE4071D,2025-08-29,2.40


In [16]:
inventory["snapshot_ts"] = pd.to_datetime(inventory["snapshot_ts"])
inventory["expiry_date"] = pd.to_datetime(inventory["expiry_date"])

inventory["days_to_expiry"] = (
    inventory["expiry_date"] - inventory["snapshot_ts"]
).dt.days 

In [17]:
velocity = (
    sales.groupby(["store_id","sku"])["qty"]
    .sum()
    .reset_index()
)

velocity["daily_velocity"] = velocity["qty"] / 15
velocity = velocity.drop(columns=["qty"])

velocity.head()

,store_id,sku,daily_velocity
0,X001,SKU00001,1.800000
1,X001,SKU00002,3.333333
2,X001,SKU00003,4.000000
3,X001,SKU00004,1.600000
4,X001,SKU00005,1.066667


In [18]:
data = inventory.merge(velocity, on=["store_id","sku"], how="left")
data["daily_velocity"] = data["daily_velocity"].fillna(0)

data.head() 

,snapshot_ts,store_id,sku,product_name,on_hand_qty,expiry_date,lot_id,purchased_date,unit_cost_at_lot,days_to_expiry,daily_velocity
0,2026-02-19 01:28:57,X001,SKU00002,TOPITOP MEYVELİ KAVANOZ 120 Lİ,24,2026-03-13,F4096A57C7ED,2025-08-05,3.17,21,3.333333
1,2026-02-19 01:28:57,X001,SKU00003,MKA 175G SOFT CHOC 16CA,48,2026-05-01,A0449612B5BF,2025-07-05,4.95,70,4.000000
2,2026-02-19 01:28:57,X001,SKU00004,Jelibon Sour Patch Kids 80 gr,144,2026-05-25,0A8610087ACD,2025-09-25,2.74,94,1.600000
3,2026-02-19 01:28:57,X001,SKU00005,Tofita Vişne 47gr,48,2026-07-05,404BA1ECEB1A,2025-09-28,2.38,135,1.066667
4,2026-02-19 01:28:57,X001,SKU00005,Tofita Vişne 47gr,36,2026-05-19,70A45BE4071D,2025-08-29,2.40,88,1.066667


In [19]:
data["coverage_days"] = data["on_hand_qty"] / (data["daily_velocity"] + 1e-6)
data["coverage_days"] = data["coverage_days"].clip(upper=365)

data[["on_hand_qty","daily_velocity","coverage_days"]].head()

,on_hand_qty,daily_velocity,coverage_days
0,24,3.333333,7.199998
1,48,4.000000,11.999997
2,144,1.600000,89.999944
3,48,1.066667,44.999958
4,36,1.066667,33.749968


In [ ]:
data["core_risk"] = (
    data["coverage_days"] /
    (data["days_to_expiry"] + 1)
)

data["stock_norm"] = ( data["on_hand_qty"] / data["on_hand_qty"].max())

data["risk_score"] = (
    0.8 * data["core_risk"] +
    0.2 * data["stock_norm"]
)
data[[
    "coverage_days", "days_to_expiry", "core_risk", "stock_norm", "risk_score"]].head()

,coverage_days,days_to_expiry,core_risk,stock_norm,risk_score
0,7.199998,21,0.327273,0.047619,0.271342
1,11.999997,70,0.169014,0.095238,0.154259
2,89.999944,94,0.947368,0.285714,0.815037
3,44.999958,135,0.330882,0.095238,0.283753
4,33.749968,88,0.379213,0.071429,0.317656


In [ ]:
def risk_bucket(x):
    if x < 0.5:
        return "low"
    elif x < 1:
        return "medium"
    else:
        return "high"

data["risk_level"] = data["risk_score"].apply(risk_bucket)

data["risk_level"].value_counts()

risk_level
low       29448
medium     3472
high       2905
Name: count, dtype: int64

In [ ]:
data.sort_values("risk_score", ascending=False).head(20)

,snapshot_ts,store_id,sku,product_name,on_hand_qty,expiry_date,lot_id,purchased_date,unit_cost_at_lot,days_to_expiry,daily_velocity,coverage_days,core_risk,stock_norm,risk_score,risk_level
12817,2026-02-19 01:28:57,X037,SKU00023,Karamelli Çikolata 65g,180,2026-02-27,8FC8BD0DC0EF,2025-08-26,1.99,7,0.733333,245.454211,30.681776,0.357143,24.616850,high
28733,2026-02-19 01:28:57,X081,SKU00117,Meyve Suyu Şeftali 200ml,48,2026-03-04,E8A807289CED,2025-07-26,3.21,12,0.000000,365.000000,28.076923,0.095238,22.480586,high
5502,2026-02-19 01:28:57,X016,SKU00172,Pekmez Üzüm 700g,481,2026-03-06,395A0DC1A591,2024-11-12,17.78,14,0.866667,365.000000,24.333333,0.954365,19.657540,high
16198,2026-02-19 01:28:57,X046,SKU00193,Nutella 750g,345,2026-03-06,C3D6809FC0B3,2025-10-21,19.16,14,1.133333,304.411496,20.294100,0.684524,16.372185,high
25940,2026-02-19 01:28:57,X073,SKU00184,Sandviç Ekmeği 400g,326,2026-03-09,FB288BA5272E,2025-10-25,10.45,17,0.266667,365.000000,20.277778,0.646825,16.351587,high
5148,2026-02-19 01:28:57,X015,SKU00145,Un Beyaz 1kg,167,2026-03-09,FDA6CF1D9545,2024-07-25,24.00,17,0.200000,365.000000,20.277778,0.331349,16.288492,high
5275,2026-02-19 01:28:57,X015,SKU00278,Kağıt Mendil Kutu 150'li,72,2026-03-09,51FEC18491A6,2025-06-03,4.83,17,0.133333,365.000000,20.277778,0.142857,16.250794,high
30465,2026-02-19 01:28:57,X086,SKU00075,Tahin Pekmez Karışımı 350g,299,2026-03-10,1504B237F04E,2024-07-05,7.21,18,0.600000,365.000000,19.210526,0.593254,15.487072,high
17125,2026-02-19 01:28:57,X049,SKU00019,Sütlü Çikolata Kare 80g,108,2026-03-05,4A784B307398,2025-04-20,2.74,13,0.400000,269.999325,19.285666,0.214286,15.471390,high
25828,2026-02-19 01:28:57,X073,SKU00078,Kraker Tuzlu 150g,108,2026-03-05,FDD1C3B67599,2025-10-09,4.08,13,0.400000,269.999325,19.285666,0.214286,15.471390,high
